In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

print("Entorno listo")
print("Versión de pandas:", pd.__version__)

Entorno listo
Versión de pandas: 3.0.5


In [2]:
# Ruta al archivo RAW
raw_path = Path("data/raw/Online Retail.xlsx")

# Cargar el dataset original
df_raw = pd.read_excel(raw_path)

print("Dataset cargado correctamente")
print("Filas:", df_raw.shape[0])
print("Columnas:", df_raw.shape[1])

df_raw.head()

Dataset cargado correctamente
Filas: 541909
Columnas: 8


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom


In [3]:
# Auditoría inicial del dataset RAW

print("Dimensiones:", df_raw.shape)

print("\n--- TIPOS DE DATOS ---")
print(df_raw.dtypes)

print("\n--- VALORES NULOS ---")
print(df_raw.isna().sum())

print("\n--- FILAS DUPLICADAS ---")
print(df_raw.duplicated().sum())

print("\n--- PERIODO DE LOS DATOS ---")
print("Fecha mínima:", df_raw["InvoiceDate"].min())
print("Fecha máxima:", df_raw["InvoiceDate"].max())

print("\n--- RESUMEN ---")
print("Facturas únicas:", df_raw["InvoiceNo"].nunique())
print("Clientes únicos:", df_raw["CustomerID"].nunique())
print("Países:", df_raw["Country"].nunique())

Dimensiones: (541909, 8)

--- TIPOS DE DATOS ---
InvoiceNo              object
StockCode              object
Description            object
Quantity                int64
InvoiceDate    datetime64[us]
UnitPrice             float64
CustomerID            float64
Country                   str
dtype: object

--- VALORES NULOS ---
InvoiceNo           0
StockCode           0
Description      1454
Quantity            0
InvoiceDate         0
UnitPrice           0
CustomerID     135080
Country             0
dtype: int64

--- FILAS DUPLICADAS ---
5268

--- PERIODO DE LOS DATOS ---
Fecha mínima: 2010-12-01 08:26:00
Fecha máxima: 2011-12-09 12:50:00

--- RESUMEN ---
Facturas únicas: 25900
Clientes únicos: 4372
Países: 38


In [4]:
# Crear una copia de trabajo
df_clean = df_raw.copy()

# Registrar tamaño antes de eliminar duplicados
rows_before = len(df_clean)

# Eliminar duplicados exactos
df_clean = df_clean.drop_duplicates().copy()

rows_after = len(df_clean)
duplicates_removed = rows_before - rows_after

print("Filas RAW:", rows_before)
print("Duplicados eliminados:", duplicates_removed)
print("Filas después de eliminar duplicados:", rows_after)

Filas RAW: 541909
Duplicados eliminados: 5268
Filas después de eliminar duplicados: 536641


In [5]:
# Crear indicadores para clasificar las transacciones

df_clean["IsCancellation"] = (
    df_clean["InvoiceNo"]
    .astype(str)
    .str.startswith("C")
)

df_clean["IsNegativeQuantity"] = df_clean["Quantity"] < 0

df_clean["IsZeroPrice"] = df_clean["UnitPrice"] == 0

df_clean["IsNegativePrice"] = df_clean["UnitPrice"] < 0

# Venta válida:
# cantidad positiva + precio positivo + factura no cancelada
df_clean["ValidSale"] = (
    (df_clean["Quantity"] > 0) &
    (df_clean["UnitPrice"] > 0) &
    (~df_clean["IsCancellation"])
)

print("--- CLASIFICACIÓN INICIAL ---")

print("Cancelaciones:",
      df_clean["IsCancellation"].sum())

print("Cantidades negativas:",
      df_clean["IsNegativeQuantity"].sum())

print("Precios en cero:",
      df_clean["IsZeroPrice"].sum())

print("Precios negativos:",
      df_clean["IsNegativePrice"].sum())

print("Líneas de venta válidas:",
      df_clean["ValidSale"].sum())

--- CLASIFICACIÓN INICIAL ---
Cancelaciones: 9251
Cantidades negativas: 10587
Precios en cero: 2510
Precios negativos: 2
Líneas de venta válidas: 524878


In [6]:
# Crear valor monetario por línea
df_clean["GrossLineValue"] = (
    df_clean["Quantity"] * df_clean["UnitPrice"]
)

# Crear variables temporales
df_clean["Year"] = df_clean["InvoiceDate"].dt.year
df_clean["Month"] = df_clean["InvoiceDate"].dt.month
df_clean["YearMonth"] = df_clean["InvoiceDate"].dt.to_period("M").astype(str)
df_clean["DayOfWeek"] = df_clean["InvoiceDate"].dt.day_name()
df_clean["Hour"] = df_clean["InvoiceDate"].dt.hour

print("Variables creadas correctamente")

print("\nNuevas dimensiones:")
print(df_clean.shape)

df_clean[
    [
        "InvoiceNo",
        "InvoiceDate",
        "Quantity",
        "UnitPrice",
        "GrossLineValue",
        "YearMonth",
        "DayOfWeek",
        "Hour",
        "ValidSale"
    ]
].head()

Variables creadas correctamente

Nuevas dimensiones:
(536641, 19)


,InvoiceNo,InvoiceDate,Quantity,UnitPrice,GrossLineValue,YearMonth,DayOfWeek,Hour,ValidSale
0,536365,2010-12-01 08:26:00,6,2.55,15.30,2010-12,Wednesday,8,True
1,536365,2010-12-01 08:26:00,6,3.39,20.34,2010-12,Wednesday,8,True
2,536365,2010-12-01 08:26:00,8,2.75,22.00,2010-12,Wednesday,8,True
3,536365,2010-12-01 08:26:00,6,3.39,20.34,2010-12,Wednesday,8,True
4,536365,2010-12-01 08:26:00,6,3.39,20.34,2010-12,Wednesday,8,True


In [7]:
# Crear subconjunto de ventas válidas
df_sales = df_clean[df_clean["ValidSale"]].copy()

print("--- VALIDACIÓN DE VENTAS ---")

print("Líneas de venta válidas:", len(df_sales))

print("Cantidades <= 0:",
      (df_sales["Quantity"] <= 0).sum())

print("Precios <= 0:",
      (df_sales["UnitPrice"] <= 0).sum())

print("Facturas canceladas:",
      df_sales["InvoiceNo"]
      .astype(str)
      .str.startswith("C")
      .sum())

print("Duplicados exactos:",
      df_clean.duplicated().sum())

print("\n--- PRIMEROS KPIs ---")

print("Ingresos brutos válidos:",
      round(df_sales["GrossLineValue"].sum(), 2))

print("Pedidos válidos:",
      df_sales["InvoiceNo"].nunique())

print("Clientes identificados:",
      df_sales["CustomerID"].nunique())

print("Unidades vendidas:",
      df_sales["Quantity"].sum())

--- VALIDACIÓN DE VENTAS ---
Líneas de venta válidas: 524878
Cantidades <= 0: 0
Precios <= 0: 0
Facturas canceladas: 0
Duplicados exactos: 0

--- PRIMEROS KPIs ---
Ingresos brutos válidos: 10642110.8
Pedidos válidos: 19960
Clientes identificados: 4338
Unidades vendidas: 5572420


In [8]:
# Exportar datasets procesados

clean_path = Path("data/clean")

# Dataset completo clasificado
df_clean.to_csv(
    clean_path / "online_retail_clean.csv",
    index=False
)

# Dataset exclusivo de ventas válidas
df_sales.to_csv(
    clean_path / "online_retail_valid_sales.csv",
    index=False
)

print("Archivos exportados correctamente")
print("Dataset CLEAN:", df_clean.shape)
print("Ventas válidas:", df_sales.shape)

Archivos exportados correctamente
Dataset CLEAN: (536641, 19)
Ventas válidas: (524878, 19)


In [9]:
from pathlib import Path

clean_path = Path("data/clean")
clean_path.mkdir(parents=True, exist_ok=True)

print("Carpeta CLEAN creada:", clean_path.exists())

Carpeta CLEAN creada: True


In [10]:
# Dataset completo clasificado
df_clean.to_csv(
    clean_path / "online_retail_clean.csv",
    index=False
)

# Dataset exclusivo de ventas válidas
df_sales.to_csv(
    clean_path / "online_retail_valid_sales.csv",
    index=False
)

print("Archivos exportados correctamente")
print("Dataset CLEAN:", df_clean.shape)
print("Ventas válidas:", df_sales.shape)

Archivos exportados correctamente
Dataset CLEAN: (536641, 19)
Ventas válidas: (524878, 19)
